In [ ]:
import sys
from pathlib import Path
import pandas as pd
print(pd.__version__)  


sys.path.insert(0, str(Path("..").resolve()))
sys.path.insert(0, "../Prop_repo/Property-based-Neural-Network-Verification/src")
import torch
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, n_features: int, num_classes: int):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = x.squeeze(1)
        return self.head(x)

import utils.models
utils.models.MLP = MLP

print(utils.models)
print(hasattr(utils.models, "MLP"))
print(utils.models.MLP)

import joblib
bundle = joblib.load("model.joblib")
print(bundle.keys())
print(type(bundle["model"]))
print(bundle.keys())

In [ ]:

import torch
import torch.nn as nn

base_model = bundle["model"]
base_model.eval()

class FlatMLPWrapper(nn.Module):
    def __init__(self, mlp):
        super().__init__()
        self.mlp = mlp

    def forward(self, x):
        return self.mlp(x)

wrapped_model = FlatMLPWrapper(base_model).eval()

n_features = len(bundle["features"])
dummy_input = torch.randn(1, n_features)

torch.onnx.export(
    wrapped_model,
    dummy_input,
    "model_pip.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=11,
    dynamo=False
)

In [ ]:
import sys
sys.path.insert(0, "/Users/karlivarkvaran/Desktop/Attacks_on_models/Marabou")
sys.path.insert(0, "/Users/karlivarkvaran/Desktop/Attacks_on_models/Marabou/build")

from maraboupy.MarabouNetworkONNX import MarabouNetworkONNX

net = MarabouNetworkONNX("formal_moda_best.onnx")
print(net)

In [ ]:
print(bundle)

classes = ["BENIGN", "HTTP_FLOOD", "PORTSCAN"]


In [ ]:
classes = ["BENIGN", "HTTP_FLOOD", "PORTSCAN"]



features = list(bundle["features"])
for i, f in enumerate(features):
    print(i, f)

In [ ]:
ATTACK_SPECS = {
    "validity": {
        "valid_packet_size_min_pkts": 1.0,
        "valid_packet_size_min_total_bytes": 40.0,
    },
    "dos_http_flood": {
        "mal_time_elapsed_min": 0.0,
        "mal_time_elapsed_max": 120.0,
        "valid_pkt_size_total_min": 250,
        "mal_byte_rate_min": 200.0,
        "mal_pkt_rate_min": 5.0,
    },
    "portscan": {
        "min_uniq_dst_ports": 20.0,
        "max_pkts_per_port": 10.0,
        "max_scan_duration": 5.0,
        "min_fail_ratio": 0.75,
    },
    "ddos_udp_flood": {
        "valid_packet_size_individual_min": 40.0,
        "valid_pkt_size_total_min": 250,
        "mal_time_elapsed_min": 0.0,
        "mal_time_elapsed_max": 120.0,
        "mal_byte_rate_min": 200.0,
        "mal_pkt_rate_min": 5.0,
    },
    "ddos_syn_flood": {
        "max_syn_duration": 10.0,
        "min_syn_conn_count": 5.0,
        "min_syn_count": 5.0,
        "min_syn_rate": 1.0,
        "min_half_open_count": 1.0,
        "min_source_ip_count": 1.0,
    },
} 

In [ ]:
RAW_BOUNDS = {
    "duration": (0.0, 120.0),
    "orig_bytes": (0.0, 200_000.0),
    "resp_bytes": (0.0, 50_000.0),
    "missed_bytes": (0.0, 1000.0),
    "orig_pkts": (0.0, 10_000.0),
    "orig_ip_bytes": (0.0, 500_000.0),
    "resp_pkts": (0.0, 5000.0),
    "resp_ip_bytes": (0.0, 200_000.0),
    "orig_pkt_rate": (0.0, 50_000.0),
    "orig_byte_rate": (0.0, 5_000_000.0),
    "time_elapsed": (0.0, 120.0),
    "valid_tcp_handshake": (0.0, 1.0),
    "valid_http_conn": (0.0, 1.0),
    "uniq_dst_ports": (1.0, 5000.0),
    "pkts_per_port": (0.0, 10_000.0),
    "scan_duration": (0.0, 120.0),
    "fail_ratio": (0.0, 1.0),
    "id.resp_p": (0.0, 65535.0),
}


In [ ]:
def scale_bound(feat, raw_val, scaler, scale_cols):
    """Convert a raw feature value to scaled space using the bundle's MinMaxScaler."""
    scale_cols = list(scale_cols)
    if feat not in scale_cols:
        return raw_val  # unscaled feature, pass through
    col_idx = scale_cols.index(feat)
    min_val = scaler.data_min_[col_idx]
    range_val = scaler.data_range_[col_idx]
    if range_val == 0:
        return 0.0
    return (raw_val - min_val) / range_val

scaler = bundle["scaler"]
scale_cols = bundle["scale_cols"]

def sb(feat, val):
    return scale_bound(feat, val, scaler, scale_cols)

In [ ]:
import sys
sys.path.insert(0, "/Users/karlivarkvaran/Desktop/Attacks_on_models/Marabou")
sys.path.insert(0, "/Users/karlivarkvaran/Desktop/Attacks_on_models/Marabou/build")

from maraboupy.MarabouNetworkONNX import MarabouNetworkONNX

scaler = bundle["scaler"]
scale_cols = list(bundle["scale_cols"])

def sb(feat, val):
    """Scale a raw value to model input space using the bundle's MinMaxScaler."""
    if feat not in scale_cols:
        return float(val)
    col_idx = scale_cols.index(feat)
    min_val = scaler.data_min_[col_idx]
    range_val = scaler.data_range_[col_idx]
    if range_val == 0:
        return 0.0
    return float((val - min_val) / range_val)

def scaled_coeff(feat):
    """Return the data_range for a feature (used to convert raw coefficients to scaled space)."""
    if feat not in scale_cols:
        return 1.0
    return float(scaler.data_range_[scale_cols.index(feat)])

def verify_portscan_property(
    model_path="model_pip.onnx",
    rival_classes=("BENIGN", "DOS_HTTP_FLOOD"),
    eps=0.1,
    print_counterexample=True,
):
    classes = ["BENIGN", "DOS_HTTP_FLOOD", "PORTSCAN"]
    features = list(bundle["features"])
    portscan_idx = classes.index("PORTSCAN")
    idx = {name: features.index(name) for name in features}

    # Directly from attack_specs["portscan"]
    # mal_uniq_dst_ports_min = 20.0
    # mal_pkts_per_port_max  = 10.0  → orig_pkts <= 10 * uniq_dst_ports
    # mal_scan_duration_max  = 5.0
    # mal_fail_ratio_min     = 0.75

    PORTSCAN_BOUNDS = {
        "duration":            (0.0,   180.0),
        "orig_pkts":           (1.0,   57.0),
        "orig_bytes":          (40.0,  6117.0),   # valid_packet_size_individual_min=40
        "orig_ip_bytes":       (44.0,  9041.0),
        "resp_pkts":           (0.0,   69.0),
        "resp_bytes":          (0.0,   85286.0),
        "resp_ip_bytes":       (0.0,   89006.0),
        "missed_bytes":        (0.0,   100.0),
        "time_elapsed":        (0.0,   120.0),
        "uniq_dst_ports":      (20.0,  999.0),    # mal_uniq_dst_ports_min=20
        "scan_duration":       (0.0,   5.0),      # mal_scan_duration_max=5
        "fail_ratio":          (0.75,  1.0),      # mal_fail_ratio_min=0.75
        "valid_tcp_handshake": (0.0,   0.0),      # no valid handshake in portscan
        "valid_http_conn":     (0.0,   0.0),      # no valid HTTP in portscan
    }

    for rival_name in rival_classes:
        rival_idx = classes.index(rival_name)
        print(f"Checking rival class: {rival_name}")

        net = MarabouNetworkONNX(model_path)
        inputs = net.inputVars[0][0]
        output_vars = net.outputVars[0][0]

        # Box bounds from attack specs
        for i, feat in enumerate(features):
            lb, ub = PORTSCAN_BOUNDS[feat]
            net.setLowerBound(inputs[i], sb(feat, lb))
            net.setUpperBound(inputs[i], sb(feat, ub))

        # orig_ip_bytes >= orig_bytes
        net.addInequality(
            [int(inputs[idx["orig_bytes"]]), int(inputs[idx["orig_ip_bytes"]])],
            [1.0/scaled_coeff("orig_bytes"), -1.0/scaled_coeff("orig_ip_bytes")],
            0.0,
        )

        # resp_ip_bytes >= resp_bytes
        net.addInequality(
            [int(inputs[idx["resp_bytes"]]), int(inputs[idx["resp_ip_bytes"]])],
            [1.0/scaled_coeff("resp_bytes"), -1.0/scaled_coeff("resp_ip_bytes")],
            0.0,
        )

        # resp_ip_bytes <= 1500 * resp_pkts (MTU)
        net.addInequality(
            [int(inputs[idx["resp_ip_bytes"]]), int(inputs[idx["resp_pkts"]])],
            [1.0/scaled_coeff("resp_ip_bytes"), -1500.0/scaled_coeff("resp_pkts")],
            0.0,
        )

        # orig_bytes >= 40 * orig_pkts (min packet size from attack spec)
        net.addInequality(
            [int(inputs[idx["orig_pkts"]]), int(inputs[idx["orig_bytes"]])],
            [40.0/scaled_coeff("orig_pkts"), -1.0/scaled_coeff("orig_bytes")],
            0.0,
        )

        # orig_pkts <= 10 * uniq_dst_ports (mal_pkts_per_port_max=10)
        net.addInequality(
            [int(inputs[idx["orig_pkts"]]), int(inputs[idx["uniq_dst_ports"]])],
            [1.0/scaled_coeff("orig_pkts"), -10.0/scaled_coeff("uniq_dst_ports")],
            0.0,
        )

        # --- Property: can rival beat PORTSCAN? ---
        net.addInequality(
            [int(output_vars[portscan_idx]), int(output_vars[rival_idx])],
            [1.0, -1.0],
            -eps,
        )
        
        # orig_pkts >= uniq_dst_ports (need at least 1 pkt per port)
        net.addInequality(
            [int(inputs[idx["uniq_dst_ports"]]), int(inputs[idx["orig_pkts"]])],
            [1.0/scaled_coeff("uniq_dst_ports"), -1.0/scaled_coeff("orig_pkts")],
            0.0,
        )

        # duration > 0 — set minimum duration
        # Already handled by box bound, but tighten it:
        # Use clip_lower for duration as minimum
        net.setLowerBound(inputs[idx["duration"]], sb("duration", 0.01))

        # orig_ip_bytes <= orig_bytes + 1500 * orig_pkts (max IP overhead per packet)
        net.addInequality(
            [int(inputs[idx["orig_ip_bytes"]]), int(inputs[idx["orig_bytes"]]), int(inputs[idx["orig_pkts"]])],
            [1.0/scaled_coeff("orig_ip_bytes"), -1.0/scaled_coeff("orig_bytes"), -1500.0/scaled_coeff("orig_pkts")],
            0.0,
        )

        print("Solving...", flush=True)
        exit_code, vals, stats = net.solve()

        if exit_code == "sat" or (isinstance(vals, dict) and len(vals) > 0):
            print(f"Counterexample found: {rival_name} beats PORTSCAN")
            if print_counterexample:
                print("\nInput feature values (scaled → raw):")
                for i, name in enumerate(features):
                    scaled_val = vals.get(inputs[i], None)
                    if scaled_val is not None and name in scale_cols:
                        col_idx = scale_cols.index(name)
                        raw_val = scaled_val * scaler.data_range_[col_idx] + scaler.data_min_[col_idx]
                    else:
                        raw_val = scaled_val
                    print(f"{i:2d} {name:30s} scaled={scaled_val!r:>10}  raw={raw_val!r}")
                print("\nOutput values:")
                for i, cls_name in enumerate(classes):
                    print(f"{i:2d} {cls_name:20s} logit={vals.get(output_vars[i], None)!r}")
            return False, rival_name, vals, stats

        elif exit_code == "unsat":
            print(f"No counterexample for rival {rival_name}")
        else:
            print(f"Solver returned: {exit_code}")

    print("No counterexample found. Property holds.")
    return True, None, None, None

In [ ]:
print(list(bundle["features"]))
verify_portscan_property(
    model_path="model_pip.onnx",
    rival_classes=("BENIGN", "DOS_HTTP_FLOOD"),
    eps = 0.1,
)



In [ ]:
HTTP_FLOOD_BOUNDS = {
    "duration":      (1.0,   180.0),   # within scaler max
    "orig_pkts":     (10.0,  57.0),    # within scaler max
    "orig_bytes":    (500.0, 6117.0),  # within scaler max
    "orig_ip_bytes": (500.0, 9041.0),  # within scaler max
    "resp_pkts":     (5.0,   69.0),    # within scaler max
    "resp_bytes":    (200.0, 85286.0), # within scaler max
    "resp_ip_bytes": (200.0, 89006.0), # within scaler max
    "missed_bytes":  (0.0,   100.0),
    "time_elapsed":  (0.0,   120.0),
    "uniq_dst_ports":(1.0,   2.0),     # HTTP flood targets 1-2 ports
    "scan_duration": (0.0,   0.05),
    "fail_ratio":    (0.0,   0.05),    # low fail ratio for HTTP flood
    "valid_tcp_handshake": (1.0, 1.0),
    "valid_http_conn":     (1.0, 1.0),
}


In [ ]:
# Fix sb() to always clamp to [0, 1]
def sb(feat, val):
    if feat not in scale_cols:
        return float(val)
    col_idx = scale_cols.index(feat)
    min_val = scaler.data_min_[col_idx]
    range_val = scaler.data_range_[col_idx]
    if range_val == 0:
        return 0.0
    scaled = float((val - min_val) / range_val)
    return max(0.0, min(1.0, scaled))  # clamp strictly to [0, 1]

In [ ]:
def verify_http_flood_property(
    model_path="model_pip.onnx",
    rival_classes=("BENIGN", "PORTSCAN"),
    eps=0.5,
    print_counterexample=True,
):
    classes = ["BENIGN", "DOS_HTTP_FLOOD", "PORTSCAN"]
    features = list(bundle["features"])
    http_idx = classes.index("DOS_HTTP_FLOOD")
    idx = {name: features.index(name) for name in features}

    HTTP_FLOOD_BOUNDS = {
        "duration":            (1.0,   180.0),
        "orig_pkts":           (10.0,  57.0),
        "orig_bytes":          (250.0, 6117.0),
        "orig_ip_bytes":       (500.0, 9041.0),
        "resp_pkts":           (5.0,   69.0),
        "resp_bytes":          (200.0, 85286.0),
        "resp_ip_bytes":       (200.0, 89006.0),
        "missed_bytes":        (0.0,   100.0),
        "time_elapsed":        (0.0,   120.0),
        "uniq_dst_ports":      (1.0,   2.0),
        "scan_duration":       (0.0,   0.05),
        "fail_ratio":          (0.0,   0.05),
        "valid_tcp_handshake": (1.0,   1.0),
        "valid_http_conn":     (1.0,   1.0),
    }

    for rival_name in rival_classes:
        rival_idx = classes.index(rival_name)
        print(f"Checking rival class: {rival_name}")

        net = MarabouNetworkONNX(model_path)
        inputs = net.inputVars[0][0]
        output_vars = net.outputVars[0][0]

        # Box bounds only — no addInequality
        for i, feat in enumerate(features):
            lb, ub = HTTP_FLOOD_BOUNDS[feat]
            net.setLowerBound(inputs[i], sb(feat, lb))
            net.setUpperBound(inputs[i], sb(feat, ub))

        # Only the output property inequality
        net.addInequality(
            [int(output_vars[http_idx]), int(output_vars[rival_idx])],
            [1.0, -1.0],
            -eps,
        )
        net.addInequality(
            [int(inputs[idx["orig_bytes"]]), int(inputs[idx["orig_ip_bytes"]])],
            [1.0/scaled_coeff("orig_bytes"), -1.0/scaled_coeff("orig_ip_bytes")],
            0.0,
        )
        
        # resp_ip_bytes >= resp_bytes  (IP bytes must be >= payload bytes)
        net.addInequality(
            [int(inputs[idx["resp_bytes"]]), int(inputs[idx["resp_ip_bytes"]])],
            [1.0/scaled_coeff("resp_bytes"), -1.0/scaled_coeff("resp_ip_bytes")],
            0.0,
        )

        # orig_ip_bytes >= orig_bytes  (same for originator)
        net.addInequality(
            [int(inputs[idx["orig_bytes"]]), int(inputs[idx["orig_ip_bytes"]])],
            [1.0/scaled_coeff("orig_bytes"), -1.0/scaled_coeff("orig_ip_bytes")],
            0.0,
        )
        
        # resp_ip_bytes <= 1500 * resp_pkts (max ethernet MTU per packet)
        net.addInequality(
            [int(inputs[idx["resp_ip_bytes"]]), int(inputs[idx["resp_pkts"]])],
            [1.0/scaled_coeff("resp_ip_bytes"), -1500.0/scaled_coeff("resp_pkts")],
            0.0,
        )

        # orig_bytes >= 100 * orig_pkts (min HTTP request size)
        net.addInequality(
            [int(inputs[idx["orig_pkts"]]), int(inputs[idx["orig_bytes"]])],
            [100.0/scaled_coeff("orig_pkts"), -1.0/scaled_coeff("orig_bytes")],
            0.0,
        )

        print("Solving...", flush=True)
        exit_code, vals, stats = net.solve()

        if exit_code == "sat" or (isinstance(vals, dict) and len(vals) > 0):
            print(f"Counterexample found: {rival_name} beats DOS_HTTP_FLOOD")
            if print_counterexample:
                print("\nInput feature values (scaled → raw):")
                for i, name in enumerate(features):
                    scaled_val = vals.get(inputs[i], None)
                    if scaled_val is not None and name in scale_cols:
                        col_idx = scale_cols.index(name)
                        raw_val = scaled_val * scaler.data_range_[col_idx] + scaler.data_min_[col_idx]
                    else:
                        raw_val = scaled_val
                    print(f"{i:2d} {name:30s} scaled={scaled_val!r:>10}  raw={raw_val!r}")
                print("\nOutput values:")
                for i, cls_name in enumerate(classes):
                    print(f"{i:2d} {cls_name:20s} logit={vals.get(output_vars[i], None)!r}")
            return False, rival_name, vals, stats

        elif exit_code == "unsat":
            print(f"No counterexample for rival {rival_name}")
        else:
            print(f"Solver returned: {exit_code}")

    print("No counterexample found. Property holds.")
    return True, None, None, None

In [ ]:
verify_http_flood_property(
    model_path="model_pip.onnx",
    rival_classes=("BENIGN","PORTSCAN"),
    eps=0.1,
)

In [ ]:
# Debug: print every bound being set for one rival
feat_bounds_debug = {}
HTTP_FLOOD_BOUNDS = {
    "duration":          (1.0,    180.0),
    "orig_pkts":         (10.0,   57.0),
    "orig_bytes":        (500.0,  6117.0),
    "orig_ip_bytes":     (500.0,  9041.0),
    "resp_pkts":         (5.0,    69.0),
    "resp_bytes":        (200.0,  85286.0),
    "resp_ip_bytes":     (200.0,  89006.0),
    "missed_bytes":      (0.0,    100.0),
    "time_elapsed":      (0.0,    120.0),
    "uniq_dst_ports":    (1.0,    2.0),
    "scan_duration":     (0.0,    0.05),
    "fail_ratio":        (0.0,    0.05),
    "valid_tcp_handshake": (1.0,  1.0),
    "valid_http_conn":     (1.0,  1.0),
}

features = list(bundle["features"])
for feat in features:
    lb, ub = HTTP_FLOOD_BOUNDS[feat]
    slb = sb(feat, lb)
    sub = sb(feat, ub)
    print(f"{feat:30s} raw=[{lb}, {ub}]  scaled=[{slb:.6f}, {sub:.6f}]  valid={'OK' if 0<=slb<=1 and 0<=sub<=1 else 'OUT OF RANGE'}")

print("\nInequality coefficients:")
print(f"  orig_bytes  coeff: {scaled_coeff('orig_bytes'):.4f}")
print(f"  orig_ip_bytes coeff: {scaled_coeff('orig_ip_bytes'):.4f}")
print(f"  resp_bytes  coeff: {scaled_coeff('resp_bytes'):.4f}")
print(f"  resp_ip_bytes coeff: {scaled_coeff('resp_ip_bytes'):.4f}")